In [3]:
# 1. Install dependencies (Run this in a Colab cell first)
# !pip install openai pandas

import os
import json
import math
import argparse
import pandas as pd
from datetime import datetime, timezone
from openai import OpenAI

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

NOW = datetime.now(timezone.utc)

# IMPORTANT: In Colab, you can also use userdata.get('api_key') for security
endpoint = "https://linked.openai.azure.com/openai/v1/"
deployment_name = "gpt-5.4-mini"
api_key = "CWcXBMalpGqTsxcKFF3movpCCR0xGwpQtMFEfIZEDTEd6oTsFtdlJQQJ99CDACYeBjFXJ3w3AAABACOGYHIY"

# ─────────────────────────────────────────────────────────────────────────────
# AZURE OPENAI CLIENT
# ─────────────────────────────────────────────────────────────────────────────

def get_client() -> OpenAI:
    return OpenAI(
        base_url=endpoint,
        api_key=api_key,
    )

def llm_score(client: OpenAI, system_prompt: str, user_content: str) -> dict:
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            temperature=0.1,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_content},
            ],
        )
        raw = response.choices[0].message.content
        return json.loads(raw)
    except Exception as e:
        print(f"Error during LLM scoring: {e}")
        return {}

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS (UNTOUCHED)
# ─────────────────────────────────────────────────────────────────────────────

def unix_to_years_ago(ts: float) -> float:
    return (NOW.timestamp() - ts) / (365.25 * 24 * 3600)

def iso_to_years_ago(iso: str) -> float:
    try:
        dt = datetime.fromisoformat(iso.replace("Z", "+00:00"))
        return (NOW - dt).days / 365.25
    except Exception:
        return 0.0

def safe(val, default=0):
    return default if (val is None or (isinstance(val, float) and math.isnan(val))) else val

def clamp(val: float, lo=0.0, hi=100.0) -> float:
    return max(lo, min(hi, val))

def stars_to_score(stars: int, max_stars: int = 5) -> float:
    return clamp((stars / max_stars) * 100)

# ─────────────────────────────────────────────────────────────────────────────
# SCORING LOGIC (KEEPING YOUR LOGIC INTACT)
# ─────────────────────────────────────────────────────────────────────────────

# ... [The score_linkedin, score_github, score_stackoverflow, score_hackerrank,
#      and score_leetcode functions remain exactly as you wrote them] ...

# [Note: I am omitting the full bodies of these 5 functions to keep this concise,
# but they should be pasted here in your actual Colab cell.]


# ─────────────────────────────────────────────────────────────────────────────
# LINKEDIN SCORER
# ─────────────────────────────────────────────────────────────────────────────

LINKEDIN_LLM_PROMPT = """
You are an expert HR analyst.  Given a LinkedIn profile (JSON), score ONLY the
following sub-metrics on a 0–100 scale (100 = best possible).  Return ONLY a
JSON object with these exact keys.  If data is missing set the value to -1.

Keys to score (LLM-based, needs interpretation):
- career_progression_trajectory   (promotions, seniority growth over time)
- network_size_quality             (connections count + follower quality)
- skill_endorsement_credibility    (breadth, relevance of skills)
- recommendation_authenticity      (count + perceived quality)
- content_quality_score            (posts, publications, featured items)
- headline_keyword_density         (technical & domain keyword richness)
- profile_completeness             (all sections filled)
- volunteer_certification_activity (certs + volunteering richness)
- company_prestige_score           (tier of employers: FAANG=100, startup=30)
- open_to_work_signal              (1 if is_job_seeker else 0, map to 0 or 100)

Return only the JSON object.  Example:
{"career_progression_trajectory": 72, "network_size_quality": 85, ...}
"""

def score_linkedin(
    profile: pd.Series,
    experience: pd.DataFrame,
    education: pd.DataFrame,
    skills: pd.DataFrame,
    client: OpenAI,
) -> dict:
    # ── Numeric sub-metrics (computed directly) ──────────────────────────────
    # Profile age (years since earliest job start)
    job_starts = []
    for _, row in experience.iterrows():
        try:
            parts = str(row["job_started_on"]).split("-")
            if len(parts) == 2:
                yr, mo = int(parts[-1]), int(parts[0])
                job_starts.append(datetime(yr, mo, 1, tzinfo=timezone.utc))
        except Exception:
            pass
    if job_starts:
        earliest = min(job_starts)
        profile_age_years = (NOW - earliest).days / 365.25
        profile_age_score = clamp(profile_age_years * 10)   # 10 yrs → 100
    else:
        profile_age_years = 0
        profile_age_score = 0.0

    # Employment consistency (avg tenure per job, months)
    if len(experience) > 0 and "experience_months" in experience.columns:
        avg_tenure = safe(experience["experience_months"].mean(), 0)
        employment_consistency_score = clamp((avg_tenure / 24) * 100)  # 24 mo ideal
    else:
        avg_tenure = 0
        employment_consistency_score = 0.0

    # Job switching frequency (jobs per year of career)
    career_span = profile_age_years if profile_age_years > 0 else 1
    jobs_per_year = len(experience) / career_span
    # fewer switches is better for this metric; invert
    job_switching_score = clamp(100 - (jobs_per_year - 1) * 20)

    # Activity frequency — use followers as proxy (no post timestamps in data)
    followers = safe(profile.get("followers", 0), 0)
    activity_score = clamp(math.log1p(followers) / math.log1p(10000) * 100)

    # Education verification (has a degree from a known university)
    has_degree = any(
        "bachelor" in str(row.get("fields_of_study", "")).lower()
        or "master" in str(row.get("fields_of_study", "")).lower()
        for _, row in education.iterrows()
    )
    education_verification_score = 100.0 if has_degree else 30.0

    numeric_scores = {
        "profile_age_years":             round(profile_age_years, 2),
        "profile_age_score":             round(profile_age_score, 1),
        "employment_consistency_score":  round(employment_consistency_score, 1),
        "avg_tenure_months":             round(avg_tenure, 1),
        "job_switching_frequency":       round(jobs_per_year, 2),
        "job_switching_score":           round(job_switching_score, 1),
        "activity_frequency_score":      round(activity_score, 1),
        "education_verification_score":  round(education_verification_score, 1),
        "connections":                   safe(profile.get("connections", 0), 0),
        "followers":                     followers,
    }

    # ── LLM sub-metrics ──────────────────────────────────────────────────────
    profile_summary = {
        "headline":               safe(profile.get("profile_headline"), ""),
        "description":            str(safe(profile.get("description"), ""))[:800],
        "skills":                 skills["skill_name"].tolist() if not skills.empty else [],
        "connections":            numeric_scores["connections"],
        "followers":              followers,
        "recommendations_given":  safe(profile.get("recommendations_given"), []),
        "recommendations_received": safe(profile.get("recommendations_received"), []),
        "certifications":         safe(profile.get("certification"), []),
        "volunteering":           safe(profile.get("volunteering"), []),
        "featured":               safe(profile.get("featured"), []),
        "is_job_seeker":          bool(safe(profile.get("is_job_seeker"), False)),
        "current_company":        safe(profile.get("current_company_name"), ""),
        "experience_roles":       experience[["job_title", "company_name", "employment_type", "experience_months"]].to_dict("records"),
        "education":              education[["university_name", "fields_of_study", "education_years"]].to_dict("records"),
    }

    llm_scores = llm_score(client, LINKEDIN_LLM_PROMPT, json.dumps(profile_summary, default=str))

    return {**numeric_scores, **llm_scores}


# ─────────────────────────────────────────────────────────────────────────────
# GITHUB SCORER
# ─────────────────────────────────────────────────────────────────────────────

GITHUB_LLM_PROMPT = """
You are a senior software engineer evaluating a GitHub profile.
Score ONLY these sub-metrics on a 0–100 scale.  Missing data → -1.

Keys (LLM-based):
- repository_originality      (% of repos that are original, not forks)
- documentation_quality       (based on repo descriptions, README presence)
- ci_cd_usage                 (evidence of CI/CD in repo topics or names)
- collaboration_network       (followers/following balance + external contributions)
- profile_readme              (does profile have a README? 0 or 100)
- language_diversity          (breadth of programming languages used)
- project_longevity           (are repos maintained long-term?)

Return only the JSON object.
"""

def score_github(
    contextual: pd.Series,
    numeric: pd.Series,
    repos: pd.DataFrame,
    client: OpenAI,
) -> dict:
    # ── Numeric sub-metrics ──────────────────────────────────────────────────
    account_age_years = iso_to_years_ago(safe(contextual.get("gh_account_created"), "2020-01-01"))
    account_age_score = clamp(account_age_years * 10)   # 10 yrs → 100

    followers = safe(numeric.get("gh_followers"), 0)
    following = safe(numeric.get("gh_following"), 1)
    public_repos = safe(numeric.get("gh_public_repos"), 0)
    total_stars = safe(numeric.get("gh_total_stars"), 0)
    forks_got = safe(numeric.get("gh_total_forks_got"), 0)
    original_repos = safe(numeric.get("gh_original_repos"), 0)

    # Commit frequency proxy: use total repos / account age
    commit_freq_proxy = clamp((public_repos / max(account_age_years, 0.5)) * 5)

    # Repository count score
    repo_count_score = clamp(math.log1p(public_repos) / math.log1p(50) * 100)

    # Stars received score
    stars_score = clamp(math.log1p(total_stars) / math.log1p(500) * 100)

    # Forks received score
    forks_score = clamp(math.log1p(forks_got) / math.log1p(100) * 100)

    # Contribution graph density proxy (no direct data; estimate from repos + age)
    contrib_density = clamp((public_repos / max(account_age_years * 365, 1)) * 30)

    numeric_scores = {
        "account_age_years":        round(account_age_years, 2),
        "account_age_score":        round(account_age_score, 1),
        "commit_frequency_score":   round(commit_freq_proxy, 1),
        "repository_count_score":   round(repo_count_score, 1),
        "stars_received_score":     round(stars_score, 1),
        "forks_received_score":     round(forks_score, 1),
        "contribution_graph_density_score": round(contrib_density, 1),
        "gh_followers":             followers,
        "gh_public_repos":          public_repos,
        "gh_total_stars":           total_stars,
        "gh_original_repos":        original_repos,
    }

    # ── LLM sub-metrics ──────────────────────────────────────────────────────
    repo_summary = repos[["repo_name", "language", "description", "stars", "forks", "topics", "updated_at"]].to_dict("records")
    ctx_summary = {
        "username":        safe(contextual.get("gh_username"), ""),
        "bio":             safe(contextual.get("gh_bio"), ""),
        "company":         safe(contextual.get("gh_company"), ""),
        "languages_used":  safe(contextual.get("gh_languages"), ""),
        "followers":       followers,
        "following":       following,
        "original_repos":  original_repos,
        "public_repos":    public_repos,
        "repos":           repo_summary[:20],   # cap at 20 for token budget
    }

    llm_scores = llm_score(client, GITHUB_LLM_PROMPT, json.dumps(ctx_summary, default=str))

    return {**numeric_scores, **llm_scores}


# ─────────────────────────────────────────────────────────────────────────────
# STACKOVERFLOW SCORER
# ─────────────────────────────────────────────────────────────────────────────

STACKOVERFLOW_LLM_PROMPT = """
You are an expert developer community analyst evaluating a Stack Overflow profile.
Score ONLY these sub-metrics 0–100.  Missing data → -1.

Keys (LLM-based):
- tag_expertise          (depth in specific tags based on top_tags)
- response_latency       (inferred; fast responders score higher — use account age vs answer count)
- top_percentage_ranking (infer from reputation rank; 0 data → -1)

Return only the JSON object.
"""

def score_stackoverflow(
    contextual: pd.Series,
    numeric: pd.Series,
    client: OpenAI,
) -> dict:
    # ── Numeric sub-metrics ──────────────────────────────────────────────────
    created_ts  = safe(numeric.get("so_account_created"), NOW.timestamp())
    account_age_years = unix_to_years_ago(created_ts)
    account_age_score = clamp(account_age_years * 10)

    reputation   = safe(numeric.get("so_reputation"), 1)
    answers      = safe(numeric.get("so_answer_count"), 0)
    questions    = safe(numeric.get("so_question_count"), 0)
    gold         = safe(numeric.get("so_gold_badges"), 0)
    silver       = safe(numeric.get("so_silver_badges"), 0)
    bronze       = safe(numeric.get("so_bronze_badges"), 0)
    accepted     = safe(numeric.get("so_accepted_answers"), 0)
    avg_score    = safe(numeric.get("so_avg_answer_score"), 0.0)

    reputation_score = clamp(math.log1p(reputation) / math.log1p(100000) * 100)

    # Answer acceptance rate
    if answers > 0:
        acceptance_rate = accepted / answers
        acceptance_score = clamp(acceptance_rate * 100)
    else:
        acceptance_rate = 0.0
        acceptance_score = 0.0

    # Answer count score
    answer_count_score = clamp(math.log1p(answers) / math.log1p(1000) * 100)

    # Question count score
    question_count_score = clamp(math.log1p(questions) / math.log1p(200) * 100)

    # Badge portfolio (weighted: gold=5, silver=2, bronze=1)
    badge_score = clamp((gold * 5 + silver * 2 + bronze) / 50 * 100)

    # Vote ratio (avg answer score)
    vote_ratio_score = clamp(math.log1p(max(avg_score, 0)) / math.log1p(50) * 100)

    numeric_scores = {
        "account_age_years":      round(account_age_years, 2),
        "account_age_score":      round(account_age_score, 1),
        "reputation_score":       round(reputation_score, 1),
        "answer_acceptance_rate": round(acceptance_rate, 3),
        "acceptance_score":       round(acceptance_score, 1),
        "answer_count_score":     round(answer_count_score, 1),
        "question_count_score":   round(question_count_score, 1),
        "badge_portfolio_score":  round(badge_score, 1),
        "vote_ratio_score":       round(vote_ratio_score, 1),
        "so_reputation":          reputation,
        "so_answer_count":        answers,
        "so_question_count":      questions,
    }

    # ── LLM sub-metrics ──────────────────────────────────────────────────────
    ctx_summary = {
        "display_name":     safe(contextual.get("so_display_name"), ""),
        "top_tags":         safe(contextual.get("so_top_tags"), ""),
        "reputation":       reputation,
        "answers":          answers,
        "accepted_answers": accepted,
        "account_age_years": account_age_years,
    }

    llm_scores = llm_score(client, STACKOVERFLOW_LLM_PROMPT, json.dumps(ctx_summary, default=str))

    return {**numeric_scores, **llm_scores}


# ─────────────────────────────────────────────────────────────────────────────
# HACKERRANK SCORER
# ─────────────────────────────────────────────────────────────────────────────

HACKERRANK_LLM_PROMPT = """
You are evaluating a HackerRank developer profile.
Score ONLY these sub-metrics 0–100.  Missing data → -1.

Keys (LLM-based):
- badge_portfolio_quality   (diversity + prestige of badges earned)
- domain_score_quality      (breadth + depth across programming domains)
- challenge_participation   (inferred activity level from available data)
- company_challenges        (evidence of company challenge participation; -1 if no data)

Return only the JSON object.
"""

def score_hackerrank(
    contextual: pd.Series,
    numeric: pd.Series,
    client: OpenAI,
) -> dict:
    # ── Numeric sub-metrics ──────────────────────────────────────────────────
    total_badges    = safe(numeric.get("hr_total_badges"), 0)
    ps_stars        = safe(numeric.get("hr_problem_solving_stars"), 0)
    python_stars    = safe(numeric.get("hr_python_stars"), 0)
    java_stars      = safe(numeric.get("hr_java_stars"), 0)
    c_stars         = safe(numeric.get("hr_c_stars"), 0)
    cpp_stars       = safe(numeric.get("hr_c++_stars"), 0)
    sql_stars       = safe(numeric.get("hr_sql_stars"), 0)
    ruby_stars      = safe(numeric.get("hr_ruby_stars"), 0)
    hr_rank         = numeric.get("hr_rank")
    hr_score        = numeric.get("hr_score")

    skill_cert_score = clamp((total_badges / 10) * 100)
    ps_score         = stars_to_score(ps_stars)
    python_score     = stars_to_score(python_stars)
    java_score       = stars_to_score(java_stars)
    cpp_score        = stars_to_score(cpp_stars)

    # Stars per domain average
    all_stars = [ps_stars, python_stars, java_stars, c_stars, cpp_stars, sql_stars, ruby_stars]
    active_domains = [s for s in all_stars if s > 0]
    avg_stars_score = clamp((sum(active_domains) / max(len(active_domains), 1)) / 5 * 100) if active_domains else 0.0

    leaderboard_score = 0.0
    if hr_rank is not None and not (isinstance(hr_rank, float) and math.isnan(hr_rank)):
        leaderboard_score = clamp(100 - (float(hr_rank) / 100000) * 100)

    score_by_domain = {
        "problem_solving": ps_score,
        "python":          python_score,
        "java":            java_score,
        "cpp":             cpp_score,
    }

    numeric_scores = {
        "skill_certificates_score": round(skill_cert_score, 1),
        "leaderboard_rank_score":   round(leaderboard_score, 1),
        "avg_stars_score":          round(avg_stars_score, 1),
        "score_by_domain":          str(score_by_domain),
        "hr_total_badges":          total_badges,
        "hr_problem_solving_stars": ps_stars,
        "hr_python_stars":          python_stars,
        "hr_java_stars":            java_stars,
        "hr_cpp_stars":             cpp_stars,
    }

    # ── LLM sub-metrics ──────────────────────────────────────────────────────
    ctx_summary = {
        "username":    safe(contextual.get("hr_username"), ""),
        "country":     safe(contextual.get("hr_country"), ""),
        "skills_raw":  safe(contextual.get("hr_skills_raw"), ""),
        "total_badges": total_badges,
        "stars_per_domain": {
            "problem_solving": ps_stars,
            "python": python_stars,
            "java": java_stars,
            "c++": cpp_stars,
            "sql": sql_stars,
            "ruby": ruby_stars,
        },
        "rank":  str(hr_rank),
        "score": str(hr_score),
    }

    llm_scores = llm_score(client, HACKERRANK_LLM_PROMPT, json.dumps(ctx_summary, default=str))

    return {**numeric_scores, **llm_scores}


# ─────────────────────────────────────────────────────────────────────────────
# LEETCODE SCORER
# ─────────────────────────────────────────────────────────────────────────────

LEETCODE_LLM_PROMPT = """
You are evaluating a LeetCode competitive programmer.
Score ONLY these sub-metrics 0–100.  Missing data → -1.

Keys (LLM-based):
- difficulty_distribution   (balance of easy/medium/hard solved)
- language_diversity        (number of languages used and their variety)
- category_coverage         (breadth of topic categories: DP, trees, graphs, etc.)
- streak_quality            (infer from contest participation count — more = better)

Return only the JSON object.
"""

def score_leetcode(
    contextual: pd.Series,
    numeric: pd.Series,
    client: OpenAI,
) -> dict:
    # ── Numeric sub-metrics ──────────────────────────────────────────────────
    total_solved    = safe(numeric.get("lc_total_solved"), 0)
    easy_solved     = safe(numeric.get("lc_easy_solved"), 0)
    medium_solved   = safe(numeric.get("lc_medium_solved"), 0)
    hard_solved     = safe(numeric.get("lc_hard_solved"), 0)
    ranking         = numeric.get("lc_ranking")
    contest_rating  = numeric.get("lc_contest_rating")
    contest_rank    = numeric.get("lc_contest_rank")
    top_pct         = numeric.get("lc_top_percentage")
    contests        = numeric.get("lc_contests_attended")
    acceptance_rate = safe(numeric.get("lc_star_rating"), 0.0)

    # Problems solved score
    solved_score = clamp(math.log1p(total_solved) / math.log1p(2500) * 100)

    # Global ranking score (lower rank = better)
    global_rank_score = 0.0
    if ranking is not None and not (isinstance(ranking, float) and math.isnan(ranking)):
        global_rank_score = clamp(100 - (float(ranking) / 3000000) * 100)

    # Contest rating score
    contest_rating_score = 0.0
    if contest_rating is not None and not (isinstance(contest_rating, float) and math.isnan(contest_rating)):
        # ~1500 is average, 2500 is high
        contest_rating_score = clamp((float(contest_rating) - 1200) / (2800 - 1200) * 100)

    # Contest participation
    contest_participation_score = 0.0
    if contests is not None and not (isinstance(contests, float) and math.isnan(contests)):
        contest_participation_score = clamp(math.log1p(float(contests)) / math.log1p(200) * 100)

    # Top percentage
    top_pct_score = 0.0
    if top_pct is not None and not (isinstance(top_pct, float) and math.isnan(top_pct)):
        top_pct_score = clamp(100 - float(top_pct))

    # Acceptance rate score
    acceptance_score = clamp(safe(acceptance_rate, 0) * 20)   # star_rating 1–5 → 20–100

    numeric_scores = {
        "problems_solved_score":          round(solved_score, 1),
        "global_ranking_score":           round(global_rank_score, 1),
        "contest_rating_score":           round(contest_rating_score, 1),
        "contest_participation_score":    round(contest_participation_score, 1),
        "top_percentage_score":           round(top_pct_score, 1),
        "acceptance_rate_score":          round(acceptance_score, 1),
        "lc_total_solved":                total_solved,
        "lc_easy_solved":                 easy_solved,
        "lc_medium_solved":               medium_solved,
        "lc_hard_solved":                 hard_solved,
        "lc_ranking":                     str(ranking),
        "lc_contest_rating":              str(contest_rating),
    }

    # ── LLM sub-metrics ──────────────────────────────────────────────────────
    ctx_summary = {
        "username":       safe(contextual.get("lc_username"), ""),
        "badges":         safe(contextual.get("lc_badges"), ""),
        "languages":      safe(contextual.get("lc_languages"), ""),
        "top_topics":     safe(contextual.get("lc_top_topics"), ""),
        "total_solved":   total_solved,
        "easy":           easy_solved,
        "medium":         medium_solved,
        "hard":           hard_solved,
        "contest_rating": str(contest_rating),
        "contests":       str(contests),
    }

    llm_scores = llm_score(client, LEETCODE_LLM_PROMPT, json.dumps(ctx_summary, default=str))

    return {**numeric_scores, **llm_scores}

# ─────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE (COLAB ADAPTED)
# ─────────────────────────────────────────────────────────────────────────────

def load(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def first_row(df: pd.DataFrame) -> pd.Series:
    return df.iloc[0] if not df.empty else pd.Series()

def run_pipeline(input_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    client = get_client()

    # Get list of files in the Colab directory
    files = {f: os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith(".csv")}

    def find(keyword: str) -> str:
        matches = [v for k, v in files.items() if keyword in k.lower()]
        return matches[0] if matches else None

    # PLATFORM MAPPING
    targets = {
        "linkedin": (find("linkedin_profile_full"), find("linkedin_experience"), find("linkedin_education"), find("linkedin_skills")),
        "github": (find("github_contextual"), find("github_numeric"), find("github_repos")),
        "stackoverflow": (find("stackoverflow_contextual"), find("stackoverflow_numeric")),
        "hackerrank": (find("hackerrank_contextual"), find("hackerrank_numeric")),
        "leetcode": (find("leetcode_contextual"), find("leetcode_numeric"))
    }

    # Process LinkedIn
    if targets["linkedin"][0]:
        print("Scoring LinkedIn...")
        li_p, li_ex, li_ed, li_sk = targets["linkedin"]
        scores = score_linkedin(first_row(load(li_p)), load(li_ex), load(li_ed), load(li_sk), client)
        pd.DataFrame([scores]).to_csv(os.path.join(output_dir, "linkedin_scores.csv"), index=False)

    # Process GitHub
    if targets["github"][0]:
        print("Scoring GitHub...")
        gh_c, gh_n, gh_r = targets["github"]
        scores = score_github(first_row(load(gh_c)), first_row(load(gh_n)), load(gh_r), client)
        pd.DataFrame([scores]).to_csv(os.path.join(output_dir, "github_scores.csv"), index=False)

    # Process StackOverflow
    if targets["stackoverflow"][0]:
        print("Scoring Stack Overflow...")
        so_c, so_n = targets["stackoverflow"]
        scores = score_stackoverflow(first_row(load(so_c)), first_row(load(so_n)), client)
        pd.DataFrame([scores]).to_csv(os.path.join(output_dir, "stackoverflow_scores.csv"), index=False)

    # Process HackerRank
    if targets["hackerrank"][0]:
        print("Scoring HackerRank...")
        hr_c, hr_n = targets["hackerrank"]
        scores = score_hackerrank(first_row(load(hr_c)), first_row(load(hr_n)), client)
        pd.DataFrame([scores]).to_csv(os.path.join(output_dir, "hackerrank_scores.csv"), index=False)

    # Process LeetCode
    if targets["leetcode"][0]:
        print("Scoring LeetCode...")
        lc_c, lc_n = targets["leetcode"]
        scores = score_leetcode(first_row(load(lc_c)), first_row(load(lc_n)), client)
        pd.DataFrame([scores]).to_csv(os.path.join(output_dir, "leetcode_scores.csv"), index=False)

    print(f"\n✅ All platform scores written to: {output_dir}")

# ─────────────────────────────────────────────────────────────────────────────
# EXECUTION CELL
# ─────────────────────────────────────────────────────────────────────────────

# In Colab, /content/ is the default local directory
INPUT_FOLDER = "/content/"
OUTPUT_FOLDER = "/content/output/"

# Just call the function directly
run_pipeline(INPUT_FOLDER, OUTPUT_FOLDER)

Scoring LinkedIn...
Scoring GitHub...
Scoring Stack Overflow...
Scoring HackerRank...
Scoring LeetCode...

✅ All platform scores written to: /content/output/
